In [ ]:
import os
import numpy as np
import pandas as pd

from dekef.base_density import *
from dekef.kernel_function import *
from dekef.data_median_dist import *

from dekef.negloglik_gubasis import *
from dekef.negloglik_scorematchingbasis import *

from dekef.plot_density_1d import *

from IPython.display import Markdown as md

import warnings
warnings.filterwarnings('ignore')
np.random.seed(1989)

In [ ]:
def negloglik_grid_grad_logpar_batchmc(data, kernel_function, base_density, coef, grid_points, batch_size, tol_param,
                                         normalizing_const_only=False, print_error=False):
    
    if len(data.shape) == 1:
        data = data.reshape(-1, 1)
    
    if len(coef.shape) == 1:
        coef = coef.reshape(-1, 1)
        
    if len(grid_points) == 1: 
        grid_points = grid_points.reshape(-1, 1)
    
    assert data.shape[1] == grid_points.shape[1], 'The dimensionality of data does not match that of grid_points.'
    
    # N, d = data.shape
    basis_n = grid_points.shape[0]

    if len(coef) != basis_n:
        raise ValueError(f"The length of coef is incorrect, which should be {basis_n} but got {len(coef)}.")
        
    if kernel_function.kernel_type == 'gaussian_poly2': 
        
        kernel_function_grid = GaussianPoly2(
            data = grid_points, 
            r1 = kernel_function.r1, 
            r2 = kernel_function.r2, 
            c = kernel_function.c, 
            bw = kernel_function.bw)
    
    elif kernel_function.kernel_type == 'rationalquad_poly2': 
        
        kernel_function_grid = RationalQuadPoly2(
            data = grid_points, 
            r1 = kernel_function.r1, 
            r2 = kernel_function.r2, 
            c = kernel_function.c, 
            bw = kernel_function.bw)
    
    ###########################################################################
    # estimate the normalizing constant 
    # first drawing
    mc_samples1 = base_density.sample(batch_size)
    mc_kernel_matrix1 = kernel_function_grid.kernel_gram_matrix(mc_samples1)
    unnorm_density_part1 = np.exp(np.matmul(mc_kernel_matrix1.T, coef))
    norm_const1 = np.mean(unnorm_density_part1)
    
    # second drawing 
    mc_samples2 = base_density.sample(batch_size)
    mc_kernel_matrix2 = kernel_function_grid.kernel_gram_matrix(mc_samples2)
    unnorm_density_part2 = np.exp(np.matmul(mc_kernel_matrix2.T, coef))
    # print(unnorm_density_part2)
    norm_const2 = np.mean(unnorm_density_part2)
    # print(norm_const2)
    
    norm_est_old = norm_const1
    norm_est_new = (norm_const1 + norm_const2) / 2
    
    error_norm = np.abs(norm_est_old - norm_est_new) / norm_est_old
    
    if print_error: 
        print('normalizing constant error = {error:.7f}'.format(error=error_norm))
    
    batch_cnt = 2
    
    while error_norm > tol_param:
        
        norm_est_old = norm_est_new
        
        # another draw
        mc_samples = base_density.sample(batch_size)
        mc_kernel_matrix = kernel_function_grid.kernel_gram_matrix(mc_samples)
        unnorm_density_part = np.exp(np.matmul(mc_kernel_matrix.T, coef))
        norm_const2 = np.mean(unnorm_density_part)
        
        # update the Monte Carlo estimation 
        norm_est_new = (norm_est_old * batch_cnt + norm_const2) / (batch_cnt + 1)

        batch_cnt += 1
        
        error_norm = np.abs(norm_est_old - norm_est_new) / norm_est_old
        
        if print_error: 
            print('normalizing constant error = {error:.7f}'.format(error=error_norm))
    
    normalizing_const = norm_est_new
    
    if not normalizing_const_only:
        
        if print_error:
            print("#" * 45 + "\nEstimating the gradient of the log-partition now.")
        
        mc_samples1 = base_density.sample(batch_size)
        mc_kernel_matrix1 = kernel_function_grid.kernel_gram_matrix(mc_samples1)
        density_part1 = np.exp(np.matmul(mc_kernel_matrix1.T, coef).flatten()) / normalizing_const
        exp_est1 = np.array([np.mean(mc_kernel_matrix1[l1, :] * density_part1)
                             for l1 in range(basis_n)]).astype(np.float64).reshape(1, -1)[0]
        
        mc_samples2 = base_density.sample(batch_size)
        mc_kernel_matrix2 = kernel_function_grid.kernel_gram_matrix(mc_samples2)
        density_part2 = np.exp(np.matmul(mc_kernel_matrix2.T, coef).flatten()) / normalizing_const
        exp_est2 = np.array([np.mean(mc_kernel_matrix2[l1, :] * density_part2)
                             for l1 in range(basis_n)]).astype(np.float64).reshape(1, -1)[0]
        
        grad_est_old = exp_est1
        grad_est_new = (exp_est1 + exp_est2) / 2
        
        error_grad = np.linalg.norm(grad_est_old - grad_est_new, 2) / (np.linalg.norm(grad_est_old, 2) * basis_n)
        
        if print_error: 
            print('gradient error = {error:.7f}'.format(error=error_grad))
        
        batch_cnt = 2
        
        while error_grad > tol_param:
        
            grad_est_old = grad_est_new

            # another draw
            mc_samples = base_density.sample(batch_size)
            mc_kernel_matrix = kernel_function_grid.kernel_gram_matrix(mc_samples)
            density_part = np.exp(np.matmul(mc_kernel_matrix.T, coef).flatten()) / normalizing_const
            exp_est2 = np.array([np.mean(mc_kernel_matrix[l1, :] * density_part)
                                 for l1 in range(basis_n)]).astype(np.float64).reshape(1, -1)[0]
            
            grad_est_new = (grad_est_old * batch_cnt + exp_est2) / (batch_cnt + 1)

            batch_cnt += 1

            error_grad = np.linalg.norm(grad_est_old - grad_est_new, 2) / (np.linalg.norm(grad_est_old, 2) * basis_n)

            if print_error: 
                print('gradient error = {error:.7f}'.format(error=error_grad))
    
    if normalizing_const_only: 
        return norm_est_new
    else: 
        return norm_est_new, grad_est_new

In [ ]:
def batch_montecarlo_params(mc_batch_size=1000, mc_tol=1e-2):
    
    """
    Returns a dictionary of parameters for the batch Monte Carlo method
    in approximating the log-partition function and its gradient.
    Parameters
    ----------
    mc_batch_size : int
        The batch size in the batch Monte Carlo method; default is 1000.
    mc_tol : float
        The floating point number below which sampling in the batch Monte Carlo is terminated; default is 1e-2.
    Returns
    -------
    dict
        The dictionary containing both supplied parameters.
    """

    mc_batch_size = int(mc_batch_size)
    
    output = {"mc_batch_size": mc_batch_size,
              "mc_tol": mc_tol}

    return output


def negloglik_optalgoparams(start_pt, step_size=0.01, max_iter=1e2, rel_tol=1e-5):
    
    """
    Returns a dictionary of parameters used in minimizing the (penalized) negative log-likelihood loss function
    by using the gradient descent algorithm.
    Parameters
    ----------
    start_pt : numpy.ndarray
        The starting point of the gradient descent algorithm to minimize
        the penalized negative log-likelihood loss function.
    step_size : float or list or numpy.ndarray
        The step size used in the gradient descent algorithm; default is 0.01.
    max_iter : int
        The maximal number of iterations in the gradient descent algorithm; default is 100.
    rel_tol : float
        The relative tolerance parameter to terminate the gradient descent algorithm in minimizing
        the penalized negative log-likelihood loss function; default is 1e-5.
    Returns
    -------
    dict
        The dictionary containing all supplied parameters.
    """

    max_iter = int(max_iter)
    
    output = {"start_pt": start_pt,
              "step_size": step_size,
              "max_iter": max_iter,
              "rel_tol": rel_tol}

    return output

In [ ]:
def negloglik_grid_coef(data, kernel_function, base_density, grid_points, lambda_param, optalgo_params, batchmc_params,
                           batch_mc=True, batch_mc_se=False, print_error=True):
    
    check_kernelfunction(kernel_function)
    check_basedensity(base_density)

    if lambda_param < 0.:
        raise ValueError("The lambda_param cannot be negative.")

    if len(data.shape) == 1:
        data = data.reshape(-1, 1)
    
    if len(grid_points.shape) == 1: 
        grid_points = grid_points.reshape(-1, 1)
    
    assert data.shape[1] == grid_points.shape[1], 'The dimensionality of data does not match that of grid_points.'
    
    N, d = data.shape
    
    if kernel_function.kernel_type == 'gaussian_poly2': 
        
        kernel_function_grid = GaussianPoly2(
            data = grid_points, 
            r1 = kernel_function.r1, 
            r2 = kernel_function.r2, 
            c = kernel_function.c, 
            bw = kernel_function.bw)
    
    elif kernel_function.kernel_type == 'rationalquad_poly2': 
        
        kernel_function_grid = RationalQuadPoly2(
            data = grid_points, 
            r1 = kernel_function.r1, 
            r2 = kernel_function.r2, 
            c = kernel_function.c, 
            bw = kernel_function.bw)

    # parameters associated with gradient descent algorithm 
    start_pt = optalgo_params["start_pt"]
    step_size = optalgo_params["step_size"]
    max_iter = optalgo_params["max_iter"]
    rel_tol = optalgo_params["rel_tol"]

    if type(step_size) not in [float, list, np.ndarray]:
        raise TypeError(("The type of step_size in optalgo_params should be one of float, list or numpy ndarray, "
                         "but got {}".format(type(step_size))))
    
    if isinstance(step_size, list) or isinstance(step_size, np.ndarray):
        warnings.warn(("The step_size you supplied in optalgo_params is a {}, "
                       "and will be reset to be the smallest positive number therein.").format(type(step_size)))
        step_size = np.array(step_size)
        step_size = np.min(step_size[step_size > 0.])

    if step_size <= 0.:
        raise ValueError("The step_size in optalgo_params must be strictly positive, but got {}.".format(step_size))
    
    if len(start_pt) != grid_points.shape[0]:
        raise ValueError(("The supplied start_pt in optalgo_params is not correct. "
                          "The expected length of start_pt is {exp_len}, but got {act_len}.").format(
            exp_len=grid_points.shape[0], act_len=len(start_pt)))
    
    # parameters associated with batch Monte Carlo estimation
    mc_batch_size = batchmc_params["mc_batch_size"]
    mc_tol = batchmc_params["mc_tol"]

    # the gradient of the loss function is
    # nabla L (alpha) = nabla A (alpha) - (1 / n) gram_matrix boldone_n + lambda_param * gram_matrix * alpha
    # the gradient descent update is
    # new_iter = current_iter - step_size * nabla L (alpha)

    # form the Gram matrix
    gram_data = kernel_function_grid.kernel_gram_matrix(data)
    f_eval = gram_data.mean(axis=1, keepdims=True)
    
    gram_grid = kernel_function_grid.kernel_gram_matrix(grid_points)
    
    current_iter = start_pt.reshape(-1, 1)

    # compute the gradient of the log-partition function at current_iter
    if batch_mc: 
                  
        mc_output1, mc_output2 = negloglik_grid_grad_logpar_batchmc(
            data=data,
            kernel_function=kernel_function,
            base_density=base_density,
            coef=current_iter,
            grid_points=grid_points,
            batch_size=mc_batch_size,
            tol_param=mc_tol,
            normalizing_const_only=False,
            print_error=False)
        
        grad_logpar = mc_output2.reshape(-1, 1)
    
    elif batch_mc_se:
        
        mc_output1, mc_output2 = negloglik_grid_grad_logpar_batchmc_se(
            data=data,
            kernel_function=kernel_function,
            base_density=base_density,
            coef=current_iter,
            grid_points=grid_points,
            batch_size=mc_batch_size,
            tol_param=mc_tol,
            normalizing_const_only=False,
            print_error=False)
        
        grad_logpar = mc_output2.reshape(-1, 1)
        
    else:
        
        raise NotImplementedError(("In order to approximate the gradient of the log-partition function, "
                                   "exactly one of 'batch_mc' and 'batch_mc_se' must be set True."))
    
    # compute the gradient of the loss function at current_iter
    current_grad = grad_logpar - f_eval + lambda_param * np.matmul(gram_grid, current_iter)
    
    # compute the updated iter
    new_iter = current_iter - step_size * current_grad
    
    # compute the error of the first update
    grad0_norm = np.linalg.norm(current_grad, 2)
    error = grad0_norm / grad0_norm
    # np.linalg.norm(new_iter - current_iter, 2) / (np.linalg.norm(current_iter, 2) + 1e-1)

    iter_num = 1

    if print_error:
        print("Iter = {iter_num}, GradNorm = {gradnorm}, Relative Error = {error}".format(
            iter_num=iter_num, gradnorm=grad0_norm, error=error))

    while error > rel_tol and iter_num < max_iter:
        
        current_iter = new_iter
        
        # compute the gradient at current_iter
        if batch_mc: 
            
            mc_output1, mc_output2 = negloglik_grid_grad_logpar_batchmc(
                data=data,
                kernel_function=kernel_function,
                base_density=base_density,
                coef=current_iter,
                grid_points=grid_points, 
                batch_size=mc_batch_size,
                tol_param=mc_tol,
                normalizing_const_only=False,
                print_error=False)
            
            grad_logpar = mc_output2.reshape(-1, 1)
        
        elif batch_mc_se:
    
            mc_output1, mc_output2 = negloglik_grid_grad_logpar_batchmc_se(
                data=data,
                kernel_function=kernel_function,
                base_density=base_density,
                coef=current_iter,
                grid_points=grid_points, 
                batch_size=mc_batch_size,
                tol_param=mc_tol,
                normalizing_const_only=False,
                print_error=False)
            
            grad_logpar = mc_output2.reshape(-1, 1)
            
        else:
            
            raise NotImplementedError(("In order to approximate the gradient of the log-partition function, "
                                       "exactly one of 'batch_mc' and 'batch_mc_se' must be set True."))

        # compute the gradient of the loss function
        current_grad = grad_logpar - f_eval + lambda_param * np.matmul(gram_grid, current_iter)

        # compute the updated iter
        new_iter = current_iter - step_size * current_grad

        # compute the error of the first update
        grad_new_norm = np.linalg.norm(current_grad, 2)
        error = grad_new_norm / grad0_norm
        # np.linalg.norm(new_iter - current_iter, 2) / (np.linalg.norm(current_iter, 2) + 1e-1)

        iter_num += 1

        if print_error:
            print("Iter = {iter_num}, GradNorm = {gradnorm}, Relative Error = {error}".format(
                iter_num=iter_num, gradnorm=grad_new_norm, error=error))

    coefficients = new_iter

    return coefficients


In [ ]:
def negloglik_grid_loss_function(data, new_data, kernel_function, base_density, coef, grid_points, batchmc_params,
                                    batch_mc=True, batch_mc_se=False):
    

    if len(data.shape) == 1:
        data = data.reshape(-1, 1)
        
    if len(grid_points.shape) == 1: 
        grid_points = grid_points.reshape(-1, 1)
    
    assert data.shape[1] == grid_points.shape[1], 'The dimensionality of data does not match that of grid_points.'
        
    N, d = data.shape
    coef = coef.reshape(-1, 1)
    if coef.shape[0] != grid_points.shape[0]:
        raise ValueError(("The supplied coef is not correct. "
                          "The expected length of coef is {exp_len}, but got {act_len}.").format(
            exp_len=grid_points.shape[0], act_len=len(coef)))
    
    mc_batch_size = batchmc_params["mc_batch_size"]
    mc_tol = batchmc_params["mc_tol"]

    if kernel_function.kernel_type == 'gaussian_poly2': 
        
        kernel_function_grid = GaussianPoly2(
            data = grid_points, 
            r1 = kernel_function.r1, 
            r2 = kernel_function.r2, 
            c = kernel_function.c, 
            bw = kernel_function.bw)
    
    elif kernel_function.kernel_type == 'rationalquad_poly2': 
        
        kernel_function_grid = RationalQuadPoly2(
            data = grid_points, 
            r1 = kernel_function.r1, 
            r2 = kernel_function.r2, 
            c = kernel_function.c, 
            bw = kernel_function.bw)
        
    # compute A(f)
    if batch_mc:
    
        mc_output1 = negloglik_grid_grad_logpar_batchmc(
            data=data,
            kernel_function=kernel_function,
            base_density=base_density,
            coef=coef,
            grid_points=grid_points,
            batch_size=mc_batch_size,
            tol_param=mc_tol,
            normalizing_const_only=True,
            print_error=False)

    elif batch_mc_se:
    
        mc_output1 = negloglik_grid_grad_logpar_batchmc_se(
            data=data,
            kernel_function=kernel_function,
            base_density=base_density,
            coef=coef,
            grid_points=grid_points,
            batch_size=mc_batch_size,
            tol_param=mc_tol,
            normalizing_const_only=True,
            print_error=False)

    else:
    
        raise NotImplementedError(("In order to approximate the gradient of the log-partition function, "
                                   "exactly one of 'batch_mc' and 'batch_mc_se' must be set True."))
    
    norm_const = mc_output1
    Af = np.log(norm_const)
    
    # compute (1 / n) sum_{j=1}^n f (Y_j), where Y_j is the j-th row of new_data
    kernel_mat_new = kernel_function_grid.kernel_gram_matrix(new_data)
    avg_fx = np.mean(np.matmul(kernel_mat_new.T, coef))
    
    loss_val = Af - avg_fx
    
    return loss_val

In [ ]:
def finer_grid_points(grid_points): 
    
    if len(grid_points.shape) == 2 and grid_points.shape[1] != 1: 
        raise ValueError(f'The function finer_grid_points only works for the 1-dimensional grid points, '
                         f'but got {grid_points.shape[1]}-dimensional.')
    
    grid_points = np.sort(grid_points.flatten())
    
    double_gp = np.array([grid_points[1:].flatten(), grid_points[:(len(grid_points) - 1)].flatten()])
    new_gp = np.mean(double_gp, axis = 0)
    output = np.sort(np.concatenate([new_gp, grid_points]))
    
    return output
        

In [ ]:
def negloglik_sieve_coef(data, kernel_function, base_density, start_grid_points, lambda_param, tol_param, 
                         optalgo_params, batchmc_params, step_size_discount_factor = 0.5, max_set_grid_points = 10, batch_mc=True, batch_mc_se=False, print_error=True):
    
    """
    Assume the starting point is just 0-vector.
    
    
    """
    
    
    # parameters associated with gradient descent algorithm 
    step_size = optalgo_params["step_size"]
    max_iter = optalgo_params["max_iter"]
    rel_tol = optalgo_params["rel_tol"]
    
    print('=' * 50)
    print('Set 1 of grid points.')
        
    optalgo_params = negloglik_optalgoparams(
        start_pt = np.zeros((start_grid_points.shape[0], 1)), 
        step_size = step_size, 
        max_iter=max_iter, 
        rel_tol=rel_tol)
    
    coef1 = negloglik_grid_coef(
        data = data, 
        kernel_function = kernel_function, 
        base_density = base_density, 
        grid_points = start_grid_points, 
        lambda_param = lambda_param, 
        optalgo_params = optalgo_params, 
        batchmc_params = batchmc_params,
        batch_mc = batch_mc, 
        batch_mc_se = batch_mc_se, 
        print_error = print_error)
    
    # evaluate the loss function 
    loss1 = negloglik_grid_loss_function(
        data = data, 
        new_data = data, 
        kernel_function = kernel_function, 
        base_density = base_density, 
        coef = coef1, 
        grid_points = start_grid_points, 
        batchmc_params = batchmc_params,
        batch_mc = batch_mc, 
        batch_mc_se = batch_mc_se)
    print(f'Negative log-likelihood loss function value = {loss1}')
    
    # second set 
    print('=' * 50)
    print('Set 2 of grid points.')
    
    new_grid_points = finer_grid_points(start_grid_points)
    optalgo_params = negloglik_optalgoparams(
        start_pt = np.zeros((new_grid_points.shape[0], 1)), 
        step_size = step_size * step_size_discount_factor, 
        max_iter=max_iter, 
        rel_tol=rel_tol)
    
    coef2 = negloglik_grid_coef(
        data = data, 
        kernel_function = kernel_function, 
        base_density = base_density, 
        grid_points = new_grid_points, 
        lambda_param = lambda_param, 
        optalgo_params = optalgo_params, 
        batchmc_params = batchmc_params,
        batch_mc = batch_mc, 
        batch_mc_se = batch_mc_se, 
        print_error = print_error)
    
    # evaluate the loss function 
    loss2 = negloglik_grid_loss_function(
        data = data, 
        new_data = data, 
        kernel_function = kernel_function, 
        base_density = base_density, 
        coef = coef2, 
        grid_points = new_grid_points, 
        batchmc_params = batchmc_params,
        batch_mc = batch_mc, 
        batch_mc_se = batch_mc_se)
    print(f'Negative log-likelihood loss function value = {loss2}')
    
    error = np.abs(loss1 - loss2) / (np.abs(loss1) + 1e-8)
    print(f'Relative improvement from increasing the number of basis functions: {error}.')
    j = 2
    
    while error > tol_param and j < max_set_grid_points: 
        
        loss1 = loss2
        
        print('=' * 50)
        print(f'Set {j + 1} of grid points.')

        new_grid_points = finer_grid_points(new_grid_points)
        optalgo_params = negloglik_optalgoparams(
            start_pt = np.zeros((new_grid_points.shape[0], 1)), 
            step_size = step_size * step_size_discount_factor, 
            max_iter=max_iter, 
            rel_tol=rel_tol)

        coef2 = negloglik_grid_coef(
            data = data, 
            kernel_function = kernel_function, 
            base_density = base_density, 
            grid_points = new_grid_points, 
            lambda_param = lambda_param, 
            optalgo_params = optalgo_params, 
            batchmc_params = batchmc_params,
            batch_mc = batch_mc, 
            batch_mc_se = batch_mc_se, 
            print_error = print_error)

        # evaluate the loss function 
        loss2 = negloglik_grid_loss_function(
            data = data, 
            new_data = data, 
            kernel_function = kernel_function, 
            base_density = base_density, 
            coef = coef2, 
            grid_points = new_grid_points, 
            batchmc_params = batchmc_params,
            batch_mc = batch_mc, 
            batch_mc_se = batch_mc_se)
        print(f'Negative log-likelihood loss function value = {loss2}')

        error = np.abs(loss1 - loss2) / (np.abs(loss1) + 1e-8)
        print(f'Relative improvement from increasing the number of basis functions: {error}.')
        
        j += 1
        
    if j == max_set_grid_points + 1: 
        print('The maximum of the sets of grid points has been reached.')
    
    return coef2


In [ ]:
os.chdir('/Users/chenxizhou/Dropbox/code_package/dekef/data')
true_data = np.load('geyser.npy').astype(np.float64)
waiting = true_data[:, 0].reshape(-1, 1)

In [ ]:
bw = 5. # bw_coef * med_dist
kernel_function = GaussianPoly2(
    data = waiting, 
    r1 = 1.0, 
    r2 = 0., 
    c = 0., 
    bw = bw)
base_density = BasedenGamma(waiting)

In [ ]:
start_grid_points = np.arange(1., 411., 10)

bmc_params = batch_montecarlo_params(
    mc_batch_size = 1000, 
    mc_tol = 1e-2)

gdalgo_params = negloglik_optalgoparams(
    start_pt = np.zeros((start_grid_points.shape[0], 1)), 
    step_size = 1.0, 
    max_iter = 50, 
    rel_tol = 1e-5)

In [ ]:
negloglik_sieve_coef(
    data = waiting, 
    kernel_function = kernel_function, 
    base_density = base_density, 
    start_grid_points = start_grid_points,
    lambda_param = 1e-5, 
    tol_param = 0.001, 
    optalgo_params = gdalgo_params, 
    batchmc_params = bmc_params, 
    step_size_discount_factor = 0.5, 
    max_set_grid_points = 3, 
    batch_mc=True, 
    batch_mc_se=False, 
    print_error=True)